In [25]:
import os
from dotenv import load_dotenv
from typing import TypedDict
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langgraph.graph import StateGraph, START, END
from huggingface_hub import login

load_dotenv()

# Authenticate with HuggingFace Token
HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [26]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    max_new_tokens=512,
    temperature=0.7
)

model = ChatHuggingFace(llm=llm)

In [27]:
# Define Pydantic Schema for Structured Output
class Evaluation(BaseModel):
    feedback: str = Field(description="Detailed feedback for the essay aspect")
    score: int = Field(description="Score out of 10 from 0 to 10", ge=0, le=10)

parser = PydanticOutputParser(pydantic_object=Evaluation)

prompt_template = PromptTemplate(
    template="""Evaluate the following essay specifically for {aspect}.
Provide constructive feedback and a score from 0 to 10.

{format_instructions}

Essay:
{essay}
""",
    input_variables=["aspect", "essay"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt_template | model | parser

In [28]:
# Define Graph State
class EssayState(TypedDict):
    essay: str
    grammar: dict
    structure: dict
    content: dict
    final_score: float
    summary: str

In [29]:
# Define Parallel Evaluator Nodes & Aggregator
def evaluate_grammar(state: EssayState):
    res = chain.invoke({"aspect": "Grammar and Vocabulary", "essay": state["essay"]})
    return {"grammar": res.model_dump()}

def evaluate_structure(state: EssayState):
    res = chain.invoke({"aspect": "Structure and Flow", "essay": state["essay"]})
    return {"structure": res.model_dump()}

def evaluate_content(state: EssayState):
    res = chain.invoke({"aspect": "Content and Argument Quality", "essay": state["essay"]})
    return {"content": res.model_dump()}

def synthesize_results(state: EssayState):
    grammar_score = state["grammar"]["score"]
    structure_score = state["structure"]["score"]
    content_score = state["content"]["score"]
    
    avg_score = round((grammar_score + structure_score + content_score) / 3.0, 2)
    
    summary = f"""=== ESSAY EVALUATION REPORT ===
Overall Score: {avg_score} / 10

1. Grammar & Vocabulary: {grammar_score}/10
   Feedback: {state['grammar']['feedback']}

2. Structure & Flow: {structure_score}/10
   Feedback: {state['structure']['feedback']}

3. Content & Arguments: {content_score}/10
   Feedback: {state['content']['feedback']}
"""
    return {"final_score": avg_score, "summary": summary}

In [30]:
# Build and Compile StateGraph Workflow
graph = StateGraph(EssayState)

# Add nodes
graph.add_node("evaluate_grammar", evaluate_grammar)
graph.add_node("evaluate_structure", evaluate_structure)
graph.add_node("evaluate_content", evaluate_content)
graph.add_node("synthesize_results", synthesize_results)

# Fan out parallel execution from START
graph.add_edge(START, "evaluate_grammar")
graph.add_edge(START, "evaluate_structure")
graph.add_edge(START, "evaluate_content")

# Fan in to aggregator node
graph.add_edge("evaluate_grammar", "synthesize_results")
graph.add_edge("evaluate_structure", "synthesize_results")
graph.add_edge("evaluate_content", "synthesize_results")

# End graph
graph.add_edge("synthesize_results", END)

workflow = graph.compile()

In [31]:
# Execute Parallel Workflow
essay_text = """Artificial Intelligence (AI) has emerged as one of the most transformative technologies of the 21st century. It has revolutionized industries ranging from healthcare to finance by enabling automated data processing and predictive insights. However, the rapid advancement of AI also brings significant challenges, including ethical dilemmas around data privacy and potential workforce displacement. To harness its full potential while mitigating risks, global policy makers and technology leaders must collaborate on robust governance frameworks."""

initial_state = {"essay": essay_text}
result = workflow.invoke(initial_state)

print(result["summary"])

=== ESSAY EVALUATION REPORT ===
Overall Score: 7.33 / 10

1. Grammar & Vocabulary: 8/10
   Feedback: The essay is well-written, but there are minor grammar and vocabulary issues. The tone is formal and suitable for an academic essay. However, there is room for improvement in sentence structure and word choice. The use of transitional phrases and connecting words can enhance the essay's flow and clarity. Overall, the essay effectively conveys the importance of AI and the need for collaborative governance. Consider revising the sentence 'To harness its full potential while mitigating risks, global policy makers and technology leaders must collaborate on robust governance frameworks.' to make it more concise and clear. Additionally, the essay could benefit from more nuanced discussion of the challenges and benefits of AI. Nevertheless, the essay is a good starting point for further development.

2. Structure & Flow: 6/10
   Feedback: The essay lacks a clear introduction and thesis stateme